# DEH 30-Day PySpark Challenge
## Day 27 — Practice Set 6: Mixed Problems

**Instructions:**
1. `File → Save a copy in Drive` first
2. Plan your approach in plain English before coding
3. Reference solutions follow each problem

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder.appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id", StringType(), True), StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True), StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True), StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", IntegerType(), True), StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True), StructField("region", StringType(), True)
])
customers_schema = StructType([
    StructField("customer_id", StringType(), True), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True), StructField("email", StringType(), True),
    StructField("city", StringType(), True), StructField("state", StringType(), True),
    StructField("country", StringType(), True), StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True)
])
products_schema = StructType([
    StructField("product_id", StringType(), True), StructField("product_name", StringType(), True),
    StructField("category", StringType(), True), StructField("sub_category", StringType(), True),
    StructField("unit_price", DoubleType(), True), StructField("cost_price", DoubleType(), True),
    StructField("supplier", StringType(), True), StructField("stock_quantity", IntegerType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

---
## Problem 1 (Medium) — Top 2 products per category by revenue

Join orders+products. Top 2 products per category by total revenue.

In [ ]:
# Plan in plain English, then your attempt


### Reference Solution — Problem 1

In [ ]:
joined = orders_df.join(products_df, on="product_id", how="inner")

product_revenue = joined.groupBy("category", "product_name").agg(
    F.round(F.sum(orders_df["unit_price"] * orders_df["quantity"]), 2).alias("total_revenue")
)

w = Window.partitionBy("category").orderBy(F.col("total_revenue").desc())

product_revenue.withColumn("rank", F.dense_rank().over(w)) \
    .filter(F.col("rank") <= 2) \
    .orderBy("category", "rank") \
    .show(20)

---
## Problem 2 (Medium) — Customers whose latest order was cancelled

Most recent order per customer; keep customers whose latest status = 'Cancelled'.

In [ ]:
# Your attempt


### Reference Solution — Problem 2

In [ ]:
w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())

latest_orders = orders_df.withColumn("rn", F.row_number().over(w)) \
    .filter(F.col("rn") == 1)

latest_orders.filter(F.col("status") == "Cancelled") \
    .select("customer_id", "order_id", "order_date", "status") \
    .show()

---
## Problem 3 (Hard) — Customer cohort analysis

Group customers by signup month. % who placed at least one order, per cohort.

In [ ]:
# Your attempt


### Reference Solution — Problem 3

In [ ]:
customers_with_cohort = customers_df.withColumn(
    "cohort_month", F.date_format("signup_date", "yyyy-MM")
)

customers_who_ordered = orders_df.select("customer_id").distinct() \
    .withColumn("has_order", F.lit(1))

cohort_status = customers_with_cohort.join(
    customers_who_ordered, on="customer_id", how="left"
).withColumn("has_order", F.coalesce(F.col("has_order"), F.lit(0)))

cohort_status.groupBy("cohort_month").agg(
    F.count("customer_id").alias("num_customers"),
    F.round(100.0 * F.sum("has_order") / F.count("customer_id"), 1).alias("pct_who_ordered")
).orderBy("cohort_month").show(20)

---
## Problem 4 (Hard) — Revenue concentration — top 20% of customers

What % of total revenue comes from the top 20% of customers by spend?

In [ ]:
# Your attempt


### Reference Solution — Problem 4

Approach: rank customers by spend, determine the row count cutoff for the top 20%, sum revenue above that cutoff, divide by total revenue.

In [ ]:
customer_spend = orders_df.groupBy("customer_id").agg(
    F.sum(F.col("unit_price") * F.col("quantity")).alias("total_spend")
).orderBy(F.col("total_spend").desc())

total_customers = customer_spend.count()
top_20_pct_count = max(1, round(total_customers * 0.20))

total_revenue = customer_spend.agg(F.sum("total_spend")).collect()[0][0]

top_customers_revenue = customer_spend.limit(top_20_pct_count) \
    .agg(F.sum("total_spend")).collect()[0][0]

pct_from_top = round(100.0 * top_customers_revenue / total_revenue, 1)

print(f"Total customers           : {total_customers}")
print(f"Top 20% count             : {top_20_pct_count}")
print(f"Total company revenue     : ${total_revenue:,.2f}")
print(f"Revenue from top 20%      : ${top_customers_revenue:,.2f}")
print(f"Percentage from top 20%   : {pct_from_top}%")

---
## Problem 5 (Hard) — Gap analysis — missing order_ids in a sequence

Order IDs should be O0001, O0002, ... sequential. Find any missing numbers.

In [ ]:
# Your attempt


### Reference Solution — Problem 5

Approach: extract the numeric part of order_id, generate the full expected sequence using `range()`, then anti-join against the actual numbers to find what's missing.

In [ ]:
# Extract numeric part of order_id
numbered = orders_df.withColumn(
    "order_num", F.substring(F.col("order_id"), 2, 10).cast("integer")
)

min_num = numbered.agg(F.min("order_num")).collect()[0][0]
max_num = numbered.agg(F.max("order_num")).collect()[0][0]

# Generate the full expected sequence
expected = spark.range(min_num, max_num + 1).withColumnRenamed("id", "expected_num")

# Anti-join — expected numbers that don't appear in actual data
missing = expected.join(
    numbered.select("order_num"),
    expected["expected_num"] == numbered["order_num"],
    how="left_anti"
)

print(f"Order number range: {min_num} to {max_num}")
print(f"Missing order numbers: {missing.count()}")
missing.orderBy("expected_num").show(20)

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*